# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### What does one row mean?

One row represents one content page for one day.

The grain for the warehouse daily table is:

report_date + client_hash_id + content_hash_id

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Which tables will I use?

Primary table:

- fact_content_daily_performance

Supporting tables:

- dim_content
- dim_clients

The daily fact table contains search and engagement measurements, while the dimension tables provide metadata and grouping information.

### Which time window?

For this assignment I will work with a mid-panel month:

2026-03

This avoids using the final month of data and follows the guidance in the lane documentation.

### What am I predictng?

I want to rank pages by refresh opportunity.

Proxy label:

trend_direction = down

Future capstone version:

Prior performance window → future decline risk

### What am I excluding?

I exclude any feature that directly contains the answer or was calculated using future information.

This helps prevent leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
!pip install duckdb

  Using cached duckdb-1.5.5-cp310-cp310-win_amd64.whl (13.2 MB)



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import duckdb

In [11]:
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


# Section 2: Query 1 (Prove the Grain)

In [12]:
duckdb.sql("""
SELECT
    trend_direction,
    COUNT(*) as pages
FROM df
GROUP BY trend_direction
""").df()

,trend_direction,pages
0,up,4388
1,down,16262
2,new,2236
3,flat,1152
4,stable,5962


# Section 3: Query 2 (Row Count and Coverage)

In [14]:
duckdb.sql("""
SELECT
    AVG(impressions_90d) as avg_impressions,
    AVG(sessions_90d) as avg_sessions
FROM df
""").df()

,avg_impressions,avg_sessions
0,5200.3663,37.066633


# Section 4: Query 3 (Availability Check)

Using a column that exists in your dataset.

In [16]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [19]:
duckdb.sql("""
SELECT
    COUNT(*) AS available_rows
FROM df
WHERE sessions_90d IS NOT NULL
""").df()

,available_rows
0,30000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## One limitation of this slice is that it uses a single snapshot of content performance and relies on a proxy decline label rather than a future outcome.

Future work should validate the approach using time-aware windows and future performance targets.

## Self-check

Before you submit, confirm each line honestly:

- [y ] Every section above is filled — markdown thinking AND the code that backs it
- [ y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ y] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.